# Revision: functions, data and control (solutions)

**Programming (5AF2029)** · Université de Neuchâtel · Dr. Selena Baset

[Course home](00_index.ipynb)

---

## Exercise 1: Change at a Swiss till

<img src="https://images.unsplash.com/photo-1724352673812-08a2aa6dc433?fm=jpg&q=70&w=800&auto=format&fit=crop" alt="A cash register" align="right" width="340" hspace="16">

**Rounding.** In Switzerland the smallest coin is 5 centimes. A total paid in cash is therefore
rounded to the nearest 5 centimes: a price of 37.42 is paid as 37.40, and a price of 263.37 as
263.35.

**Notes and coins.** The change is given with these denominations:

- notes: 1000, 200, 100, 50, 20 and 10 francs
- coins: 5, 2, 1, 0.50, 0.20, 0.10 and 0.05 francs

**Fewest pieces.** The change should use as few pieces as possible. For these denominations, this
is achieved by always giving the largest denomination that does not exceed the amount still owed.

<sub>Picture: Guick on <a href="https://unsplash.com/photos/a-blue-cash-register-sitting-on-top-of-a-counter-0bTqlUEmkrY">Unsplash</a>, Unsplash License.</sub>

**What to do**

1. Write a function `make_change(price, paid)` that returns the change as a dictionary.

   - The keys are the denominations in francs, and the values are the number of pieces of each.
   - Denominations that are not used do not appear in the dictionary.
   - If the payment does not cover the rounded price, the function returns `None`.
   - The change must be exact to the centime.

2. The list `sales` below holds one day's cash sales, each as a price and the amount handed over.
   Process the whole day. A sale where the customer did not hand over enough is refused, and the
   till moves on to the next sale. Report:

   - the total change handed out;
   - the number of pieces given;
   - the number of refused sales;
   - the denomination handed out most often.

*Adapted from the problem "Cash", Problem Set 1, CS50, Harvard University.*

> **Hint:** for the day's sales, the total change handed out is 171.55, 35 pieces are given,
> 2 sales are refused, and the denomination handed out most often is the 0.20 coin, 7 times.

In [1]:
sales = [[18.65, 20], [7.42, 10], [133.80, 200], [4.99, 5], [61.17, 50],
         [29.90, 50], [2.36, 10], [88.03, 100], [15.00, 15], [245.55, 300],
         [9.81, 5], [42.74, 50]]

In [2]:
# Task 1: the function make_change

# The notes and coins, in francs, from the largest to the smallest.
# The order matters: the change is built by always trying the largest piece first.
denominations = [1000, 200, 100, 50, 20, 10, 5, 2, 1, 0.50, 0.20, 0.10, 0.05]


def to_centimes(amount):
    """Convert an amount in francs to a whole number of centimes."""
    # The computer does not store amounts such as 0.29 exactly:
    # 0.29 * 100 gives 28.999999999999996 instead of 29.
    # round() turns this into the whole number 29, which is exact.
    # int() would cut it down to 28, which is why round() is used here.
    centimes = round(amount * 100)
    return centimes


def round_to_5_centimes(centimes):
    """Round a number of centimes to the nearest multiple of 5."""
    # Example with 3742 centimes:
    # 3742 / 5 = 748.4, which rounds to 748, and 748 * 5 = 3740 centimes.
    number_of_fives = round(centimes / 5)
    return number_of_fives * 5


def make_change(price, paid):
    """Return the change as {denomination in francs: number of pieces}, or None."""
    # Step 1: convert both amounts to centimes, and round the price to 5 centimes.
    # From here on, every calculation uses whole numbers, so the result is exact.
    price_in_centimes = round_to_5_centimes(to_centimes(price))
    paid_in_centimes = to_centimes(paid)

    # Step 2: refuse the sale if the customer did not pay enough.
    if paid_in_centimes < price_in_centimes:
        return None

    # Step 3: the change still to give, in centimes.
    remaining = paid_in_centimes - price_in_centimes

    # Step 4: go through the denominations, from the largest to the smallest.
    change = {}
    for denomination in denominations:
        value = to_centimes(denomination)

        # Give this piece as many times as it fits in the change still to give.
        count = 0
        while remaining >= value:
            remaining = remaining - value
            count = count + 1

        # Only record the denominations that were used.
        if count > 0:
            change[denomination] = count

    return change


# Checks

def show_change(price, paid):
    """Print the change for one payment, one denomination per line."""
    print()
    print("Price", price, "paid with", paid)

    change = make_change(price, paid)

    if change is None:
        print("   refused: the payment does not cover the price")
    elif len(change) == 0:
        # An empty dictionary: the customer paid the exact amount.
        print("   no change to give")
    else:
        for denomination in change:
            print("  ", change[denomination], "x", denomination)


print("Task 1: checks")
show_change(37.42, 100)
show_change(263.37, 300)
show_change(0.98, 5)
show_change(12.30, 12.30)
show_change(49.99, 20)

Task 1: checks

Price 37.42 paid with 100
   1 x 50
   1 x 10
   1 x 2
   1 x 0.5
   1 x 0.1

Price 263.37 paid with 300
   1 x 20
   1 x 10
   1 x 5
   1 x 1
   1 x 0.5
   1 x 0.1
   1 x 0.05

Price 0.98 paid with 5
   2 x 2

Price 12.3 paid with 12.3
   no change to give

Price 49.99 paid with 20
   refused: the payment does not cover the price


In [3]:
# Task 2: the day's sales

# Totals for the day. They start at 0 and grow with each sale.
total_change = 0      # in centimes, so that the sum stays exact
total_pieces = 0
refused = 0

# The number of pieces handed out for each denomination.
# Example: {20: 3, 0.5: 4} would mean three 20-franc notes and four 50-centime coins.
handed_out = {}

for sale in sales:
    # Each sale is a list of two numbers: the price, then the amount handed over.
    price = sale[0]
    paid = sale[1]

    change = make_change(price, paid)

    if change is None:
        # The customer did not pay enough: count the refusal and go on to the next sale.
        refused = refused + 1
    else:
        for denomination in change:
            count = change[denomination]

            total_change = total_change + to_centimes(denomination) * count
            total_pieces = total_pieces + count

            # Add the pieces to the count for this denomination.
            # The first time a denomination is handed out, it is not yet a key of the dictionary.
            if denomination in handed_out:
                handed_out[denomination] = handed_out[denomination] + count
            else:
                handed_out[denomination] = count

# Find the denomination handed out most often.
# most_often and most_count hold the best denomination found so far and its count.
most_often = None
most_count = 0

for denomination in handed_out:
    if handed_out[denomination] > most_count:
        most_often = denomination
        most_count = handed_out[denomination]

print("Task 2: the day's sales")
print()
print("   Total change handed out:", total_change / 100, "francs")
print("   Pieces given:           ", total_pieces)
print("   Sales refused:          ", refused)
print("   Handed out most often:  ", most_often, "francs,", most_count, "times")

Task 2: the day's sales

   Total change handed out: 171.55 francs
   Pieces given:            35
   Sales refused:           2
   Handed out most often:   0.2 francs, 7 times


---

## Exercise 2: Federal, cantonal and communal tax

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Switzerland_Cantons_Map_with_Names_and_Capitals_%28french%29.svg?width=800" alt="Map of the Swiss cantons" align="right" width="380" hspace="16">

**Three levels of tax.** In Switzerland, income is taxed by the Confederation, by the canton and
by the commune.

- **Federal tax:** follows its own progressive scale, and is never more than 11.5% of the income.
- **Cantonal tax:** the canton's progressive scale gives a *basic tax*, which is then multiplied by
  the cantonal multiplier.
- **Communal tax:** the same basic tax, multiplied by the commune's own multiplier.

**Why communes differ.** Two taxpayers with the same income in the same canton can pay different
amounts of tax, depending on their commune. Comparison services such as the one offered by Comparis
rank communes by this tax burden.

**How a progressive scale works.** Each bracket covers the income above the upper limit of the
previous bracket, up to and including its own upper limit. Its rate applies only to the part of the
income that falls inside it. The last bracket has no upper limit, which is marked with `None`.

For an income of 45,000, the basic scale below gives:

| Part of the income | Rate | Tax |
|---|---|---|
| first 10,000 | 0% | 0 |
| next 20,000 | 2% | 400 |
| next 15,000 | 4% | 600 |
| **total** | | **1,000** |

**About the figures.** The scales and multipliers are fictional, but the calculation follows the
Swiss structure. The incomes are taxable incomes, after all deductions, and church tax is left
out.

<sub>Picture: the 26 cantons of Switzerland, map by Poulpy, <a href="https://creativecommons.org/licenses/by/3.0/">CC BY 3.0</a>, via Wikimedia Commons.</sub>

**What to do**

1. Write a function `progressive_tax(income, scale)` that returns the tax for an income under a
   progressive scale, rounded to the centime. The same function must serve for both scales.

2. Write a function `tax_breakdown(income, commune)` that returns a dictionary with the federal
   tax, the cantonal tax, the communal tax and the total, each rounded to the centime.

   - The federal tax must respect the 11.5% limit.
   - An income cannot be negative. If it is, the function rejects it: it prints the message
     `An income cannot be negative.` and returns `None`.
   - In the same way, if the commune is not in `communal_multipliers`, the function prints a
     message saying the commune is unknown and returns `None`.

3. For an income of 95,000:

   - print the total tax in each commune;
   - report the commune where the tax is lowest and the commune where it is highest;
   - report how much this taxpayer saves each year by living in the first rather than the second.

4. The **marginal rate** is the share of the next franc earned that goes to tax, at the three
   levels together. It is the federal rate that applies to that franc, plus the basic rate that
   applies to it multiplied by both the cantonal and the communal multiplier.

   Write a function `marginal_rate(income, commune)`. Then, for an income of 95,000 in Clairval,
   print in percent:

   - the average rate, which is the total tax divided by the income;
   - the marginal rate.

5. An employee living in Clairval earns 99,500 and is offered a raise of 1,000. The raise takes her
   income past 100,000, where both the federal and the basic scale move to a higher bracket.

   A colleague warns her that she will lose money, because "all her income will now be taxed at
   the higher rates". Compute by how much her total tax rises and how much of the raise she keeps.
   Then explain why the colleague is wrong.

*The comparison of communes is inspired by the Comparis tax calculator, and the progressive scale
by the income tax exercise in Fedor Iskhakov's Computational Economics course and r/dailyprogrammer
challenge 379, "Progressive taxation".*

> **Hint:** for an income of 95,000 in Clairval, the federal tax is 3,500, the cantonal tax
> 4,625 and the communal tax 2,627, a total of 10,752. For an income of 1,000,000, the federal tax
> is 115,000 in every commune, because of the 11.5% limit.

In [4]:
federal_scale = [
    {"up_to": 18000, "rate": 0.00},
    {"up_to": 33000, "rate": 0.01},
    {"up_to": 58000, "rate": 0.03},
    {"up_to": 76000, "rate": 0.06},
    {"up_to": 100000, "rate": 0.08},
    {"up_to": 150000, "rate": 0.11},
    {"up_to": None, "rate": 0.13},
]
federal_limit = 0.115

basic_scale = [
    {"up_to": 10000, "rate": 0.00},
    {"up_to": 30000, "rate": 0.02},
    {"up_to": 60000, "rate": 0.04},
    {"up_to": 100000, "rate": 0.06},
    {"up_to": 200000, "rate": 0.08},
    {"up_to": None, "rate": 0.09},
]
cantonal_multiplier = 1.25

communal_multipliers = {
    "Lacville": 0.62,
    "Hautmont": 0.79,
    "Valrose": 0.68,
    "Pierrefeu": 0.85,
    "Clairval": 0.71,
    "Boisrond": 0.57,
}

In [5]:
# Task 1: the function progressive_tax

def progressive_tax(income, scale):
    """Return the tax on an income under a progressive scale, rounded to the centime."""
    tax = 0

    # lower is where the current bracket starts. The first bracket starts at 0.
    lower = 0

    # The brackets are listed from the lowest to the highest, so the loop
    # goes up through the income one bracket at a time.
    for bracket in scale:
        upper = bracket["up_to"]
        rate = bracket["rate"]

        # The last bracket has no upper limit (None), so the income always stops there.
        # upper is tested for None first: comparing a number with None would cause an error.
        if upper is None or income <= upper:
            # The income stops inside this bracket.
            # Only the part of the income above lower is taxed at this rate.
            tax = tax + (income - lower) * rate

            # The higher brackets do not apply, so the loop stops here.
            break
        else:
            # The income goes past this bracket.
            # The whole bracket, from lower to upper, is taxed at this rate.
            tax = tax + (upper - lower) * rate

            # The next bracket starts where this one ends.
            lower = upper

    return round(tax, 2)


# Check with the example of the table above: 45,000 under the basic scale gives 1,000.
print("Task 1: tax on 45,000 under the basic scale:", progressive_tax(45000, basic_scale))

Task 1: tax on 45,000 under the basic scale: 1000.0


In [6]:
# Task 2: the function tax_breakdown

def tax_breakdown(income, commune):
    """Return the federal, cantonal, communal and total tax as a dictionary, or None."""
    # Reject the inputs that make no sense.
    if income < 0:
        print("An income cannot be negative.")
        return None

    if commune not in communal_multipliers:
        print("Unknown commune:", commune)
        return None

    # Federal tax: its own scale, but never more than 11.5% of the income.
    federal = progressive_tax(income, federal_scale)
    limit = round(federal_limit * income, 2)
    if federal > limit:
        federal = limit

    # Cantonal and communal tax: the same basic tax, each multiplied by its own multiplier.
    basic = progressive_tax(income, basic_scale)
    cantonal = round(basic * cantonal_multiplier, 2)
    communal = round(basic * communal_multipliers[commune], 2)

    total = round(federal + cantonal + communal, 2)

    breakdown = {
        "federal": federal,
        "cantonal": cantonal,
        "communal": communal,
        "total": total,
    }
    return breakdown


# Checks

def print_breakdown(income, commune):
    """Print the tax breakdown for an income and a commune, one level per line."""
    print()
    print("Income", income, "in", commune)

    breakdown = tax_breakdown(income, commune)

    # tax_breakdown returns None for a negative income or an unknown commune.
    # In that case it has already printed a message, and there is nothing else to print.
    if breakdown is not None:
        print("   federal: ", breakdown["federal"])
        print("   cantonal:", breakdown["cantonal"])
        print("   communal:", breakdown["communal"])
        print("   total:   ", breakdown["total"])


print("Task 2: checks")
print_breakdown(95000, "Clairval")
print_breakdown(1000000, "Lacville")     # the federal tax is limited to 11.5% of 1,000,000
print_breakdown(-500, "Clairval")        # rejected: negative income
print_breakdown(95000, "Neuchâtel")      # rejected: not in communal_multipliers

Task 2: checks

Income 95000 in Clairval
   federal:  3500.0
   cantonal: 4625.0
   communal: 2627.0
   total:    10752.0

Income 1000000 in Lacville
   federal:  115000.0
   cantonal: 105000.0
   communal: 52080.0
   total:    272080.0

Income -500 in Clairval
An income cannot be negative.

Income 95000 in Neuchâtel
Unknown commune: Neuchâtel


In [7]:
# Task 3: comparing the communes

income = 95000

# The commune with the lowest tax found so far, and its tax. Same for the highest.
# They start as None, because no commune has been looked at yet.
lowest_commune = None
lowest_total = None
highest_commune = None
highest_total = None

print("Task 3: total tax for an income of 95,000")
print()

for commune in communal_multipliers:
    breakdown = tax_breakdown(income, commune)
    total = breakdown["total"]
    print("  ", commune, total)

    # The first commune is automatically the lowest so far.
    # After that, a commune replaces it only if its tax is lower.
    if lowest_commune is None:
        lowest_commune = commune
        lowest_total = total
    elif total < lowest_total:
        lowest_commune = commune
        lowest_total = total

    # The same for the highest tax.
    if highest_commune is None:
        highest_commune = commune
        highest_total = total
    elif total > highest_total:
        highest_commune = commune
        highest_total = total

saving = highest_total - lowest_total

print()
print("   Lowest tax: ", lowest_commune)
print("   Highest tax:", highest_commune)
print("   Yearly saving by living in", lowest_commune, "rather than", highest_commune + ":",
      round(saving, 2))

Task 3: total tax for an income of 95,000

   Lacville 10419.0
   Hautmont 11048.0
   Valrose 10641.0
   Pierrefeu 11270.0
   Clairval 10752.0
   Boisrond 10234.0

   Lowest tax:  Boisrond
   Highest tax: Pierrefeu
   Yearly saving by living in Boisrond rather than Pierrefeu: 1036.0


In [8]:
# Task 4: average and marginal rate

def bracket_rate(income, scale):
    """Return the rate of the bracket in which the next franc of income falls."""
    for bracket in scale:
        # The test uses < and not <=.
        # Example: at an income of exactly 100,000, the franc already earned is in the
        # bracket that ends at 100,000, but the next franc falls in the bracket above it.
        if bracket["up_to"] is None or income < bracket["up_to"]:
            return bracket["rate"]


def marginal_rate(income, commune):
    """Return the share of the next franc earned that goes to tax at the three levels."""
    # The next franc falls in one bracket of the federal scale and one of the basic scale.
    federal = bracket_rate(income, federal_scale)
    basic = bracket_rate(income, basic_scale)

    # The next franc pays:
    # - the federal rate,
    # - plus the basic rate multiplied by the cantonal multiplier,
    # - plus the basic rate multiplied by the communal multiplier.
    return federal + basic * cantonal_multiplier + basic * communal_multipliers[commune]


income = 95000
commune = "Clairval"

breakdown = tax_breakdown(income, commune)

# The average rate: the share of the whole income that goes to tax.
average = breakdown["total"] / income

# The marginal rate: the share of the next franc that goes to tax.
marginal = marginal_rate(income, commune)

print("Task 4: rates for an income of 95,000 in Clairval")
print()
print("   Average rate: ", round(average * 100, 2), "%")
print("   Marginal rate:", round(marginal * 100, 2), "%")

# The marginal rate is higher than the average rate.
# The lower parts of the income are taxed at lower rates, some at 0%,
# and they pull the average down. The next franc is taxed at the rates of the top brackets reached.

Task 4: rates for an income of 95,000 in Clairval

   Average rate:  11.32 %
   Marginal rate: 19.76 %


In [9]:
# Task 5: the raise

tax_before = tax_breakdown(99500, "Clairval")["total"]
tax_after = tax_breakdown(100500, "Clairval")["total"]

increase = tax_after - tax_before
kept = 1000 - increase

print("Task 5: a raise of 1,000 from 99,500 in Clairval")
print()
print("   Tax before the raise:", tax_before)
print("   Tax after the raise: ", tax_after)
print("   Tax increase:        ", round(increase, 2))
print("   Kept from the raise: ", round(kept, 2))

# Why the colleague is wrong:
# a higher rate applies only to the part of the income inside its bracket.
# - The first 500 of the raise take the income from 99,500 to 100,000.
#   They stay in the same brackets and are taxed as before, at 19.76% in total.
# - The other 500 take the income from 100,000 to 100,500.
#   Only these 500 are taxed at the higher rates, 26.68% in total.
# The first 100,000 of her income are taxed exactly as before the raise.
# So the tax rises by much less than the raise, and she keeps most of it.

Task 5: a raise of 1,000 from 99,500 in Clairval

   Tax before the raise: 11641.2
   Tax after the raise:  11873.4
   Tax increase:         232.2
   Kept from the raise:  767.8


---

## Exercise 3: Investing with hindsight

<img src="https://images.unsplash.com/photo-1767424196045-030bbde122a4?fm=jpg&q=70&w=800&auto=format&fit=crop" alt="A tablet showing a stock market chart" align="right" width="340" hspace="16">

**The data.** The list `prices` below holds the closing price of a share on 20 consecutive trading
days. Day 1 is the first item of the list.

**Trading.** A trade means buying shares at the closing price of one day and selling them at the
closing price of a later day.

**The investor.** An investor starts with CHF 5,000 and follows three rules:

- only whole shares can be bought;
- every order, whether buying or selling, costs a fee of CHF 25, paid from the cash;
- when buying, the fee is paid first, and the rest of the cash buys as many whole shares as
  possible. The money left over stays as cash.

<sub>Picture: Jakub Żerdzicki on <a href="https://unsplash.com/photos/hand-holding-tablet-showing-stock-market-candlestick-chart-h_H1KtjNOAk">Unsplash</a>, Unsplash License.</sub>

**What to do**

1. Write a function `best_trade(prices)` that finds the single trade with the largest profit per
   share. It returns a dictionary with the buy day, the sell day and the profit per share, or
   `None` if no trade makes a profit.

2. Write a function `largest_drop(prices)` that finds the worst possible trade: the largest fall
   from the price on one day to the price on a later day. It returns a dictionary with the day of
   the high price, the day of the low price, the loss per share, and the loss in percent of the
   high price. It returns `None` if the price never falls.

3. Compute how much money the investor has at the end of day 20 with each of these strategies:

   - **Buy and hold:** buy on day 1 and sell on day 20.
   - **Best trade:** buy and sell on the days found in task 1.
   - **Every rise:** go through the days in order. If no shares are held and the next day's price
     is higher, buy. If shares are held and the next day's price is lower, or if it is the last
     day, sell.

4. The every-rise strategy places many more orders than the others. From which fee per order, in
   whole francs, does it stop doing better than the best trade?

5. The best trade and the every-rise strategy both use information that no investor has on the
   day of the decision. Say which information this is, and what these two results can still be
   useful for.

*Tasks 1 and 3 are adapted from LeetCode problems 121 and 122, "Best Time to Buy and Sell
Stock" I and II.*

> **Hint:** with the buy-and-hold strategy, the investor ends day 20 with 5,120.00.

In [10]:
prices = [142.5, 139.8, 145.2, 151.0, 148.3, 144.9, 138.6, 141.2, 147.8, 153.4,
          150.1, 146.7, 136.2, 133.9, 140.5, 144.0, 142.2, 149.6, 152.8, 147.5]

In [11]:
# Task 1: the function best_trade

# Reminder: the list starts at position 0, so day 1 is prices[0], day 2 is prices[1], and so on.

def best_trade(prices):
    """Return the trade with the largest profit per share, or None."""
    # The best trade found so far, and its profit.
    # best_profit starts at 0, so that only a real profit can replace it.
    best = None
    best_profit = 0

    # The position of the lowest price seen so far. The first day is the lowest at the start.
    lowest_day = 0

    # Go through the days from the second one, since a sale must come after a purchase.
    for day in range(1, len(prices)):
        # If today's price is lower than any price before, buying today is the best choice
        # for any later sale.
        if prices[day] < prices[lowest_day]:
            lowest_day = day

        # The profit from buying at the lowest price so far and selling today.
        profit = prices[day] - prices[lowest_day]

        # Keep this trade if it beats the best one found so far.
        # Days are counted from 1, so 1 is added to each position.
        if profit > best_profit:
            best_profit = profit
            best = {
                "buy": lowest_day + 1,
                "sell": day + 1,
                "profit": round(profit, 2),
            }

    return best


trade = best_trade(prices)

print("Task 1: best trade")
print()
print("   Buy on day", trade["buy"], "and sell on day", trade["sell"])
print("   Profit per share:", trade["profit"])
print()
print("   Check with prices that only fall:", best_trade([50, 48, 47.5, 45]))

# The best trade is not "buy at the lowest price, sell at the highest price":
# the highest price (day 10) comes before the lowest price (day 14).

Task 1: best trade

   Buy on day 14 and sell on day 19
   Profit per share: 18.9

   Check with prices that only fall: None


In [12]:
# Task 2: the function largest_drop

# The same idea as best_trade, the other way round:
# keep the highest price seen so far, and look for the largest fall after it.

def largest_drop(prices):
    """Return the largest fall from one day's price to a later day's price, or None."""
    worst = None
    worst_loss = 0

    highest_day = 0

    for day in range(1, len(prices)):
        if prices[day] > prices[highest_day]:
            highest_day = day

        # The loss from buying at the highest price so far and selling today.
        loss = prices[highest_day] - prices[day]

        if loss > worst_loss:
            worst_loss = loss
            percent = loss / prices[highest_day] * 100
            worst = {
                "high": highest_day + 1,
                "low": day + 1,
                "loss": round(loss, 2),
                "percent": round(percent, 2),
            }

    return worst


drop = largest_drop(prices)

print("Task 2: largest drop")
print()
print("   From day", drop["high"], "to day", drop["low"])
print("   Loss per share:", drop["loss"])
print("   Loss in percent of the high price:", drop["percent"], "%")
print()
print("   Check with prices that only rise:", largest_drop([10, 11, 12]))

Task 2: largest drop

   From day 10 to day 14
   Loss per share: 19.5
   Loss in percent of the high price: 12.71 %

   Check with prices that only rise: None


In [13]:
# Task 3: three strategies

def buy(cash, price, fee):
    """Pay the fee, buy as many whole shares as possible, and return the shares and the cash left."""
    # int() drops the decimals, which gives the number of whole shares that can be paid for.
    # Example: 4975 / 142.5 = 34.9..., so 34 shares.
    shares = int((cash - fee) / price)
    cash_left = round(cash - fee - shares * price, 2)

    # A function can only return one value, so the two results are put in a dictionary.
    return {"shares": shares, "cash": cash_left}


def sell(cash, shares, price, fee):
    """Sell all the shares, pay the fee, and return the cash."""
    return round(cash + shares * price - fee, 2)


def buy_and_hold(prices, cash, fee):
    """Buy on the first day, sell on the last day, and return the cash at the end."""
    position = buy(cash, prices[0], fee)

    # prices[-1] is the last item of the list, that is, the price on the last day.
    return sell(position["cash"], position["shares"], prices[-1], fee)


def best_single_trade(prices, cash, fee):
    """Buy and sell on the days of the best trade, and return the cash at the end."""
    trade = best_trade(prices)

    # best_trade counts days from 1, so 1 is subtracted to get the position in the list.
    buy_price = prices[trade["buy"] - 1]
    sell_price = prices[trade["sell"] - 1]

    position = buy(cash, buy_price, fee)
    return sell(position["cash"], position["shares"], sell_price, fee)


def every_rise(prices, cash, fee):
    """Buy before each rise, sell before each fall, and return the cash at the end."""
    shares = 0
    last_day = len(prices) - 1

    for day in range(len(prices)):
        today = prices[day]

        if day == last_day:
            # On the last day, sell the shares that are still held.
            if shares > 0:
                cash = sell(cash, shares, today, fee)
                shares = 0
        else:
            tomorrow = prices[day + 1]

            if shares == 0 and tomorrow > today:
                # No shares held, and the price is about to rise: buy.
                position = buy(cash, today, fee)
                shares = position["shares"]
                cash = position["cash"]
            elif shares > 0 and tomorrow < today:
                # Shares held, and the price is about to fall: sell.
                cash = sell(cash, shares, today, fee)
                shares = 0

    return cash


print("Task 3: money at the end of day 20, starting with 5,000 and a fee of 25 per order")
print()
print("   Buy and hold:", buy_and_hold(prices, 5000, 25))
print("   Best trade:  ", best_single_trade(prices, 5000, 25))
print("   Every rise:  ", every_rise(prices, 5000, 25))

Task 3: money at the end of day 20, starting with 5,000 and a fee of 25 per order

   Buy and hold: 5120.0
   Best trade:   5649.3
   Every rise:   6644.5


In [14]:
# Task 4: the fee from which every rise stops doing better

# Start with no fee, and raise it by 1 franc as long as every rise still does at least
# as well as the best trade. The loop stops at the first fee where it does worse.
fee = 0
while every_rise(prices, 5000, fee) >= best_single_trade(prices, 5000, fee):
    fee = fee + 1

print("Task 4: every rise stops doing better than the best trade from a fee of", fee, "per order")

# The every-rise strategy places 8 orders, against 2 for the best trade.
# The fee has to be very high before these extra orders cost more than they bring.

Task 4: every rise stops doing better than the best trade from a fee of 163 per order


In [15]:
# Task 5: what these results are useful for

# Both strategies use the future prices:
# - the best trade needs to know on day 14 that no lower price will follow;
# - the every-rise strategy needs to know each next day's price before it happens.
# No investor has this information on the day of the decision, so neither strategy can be followed.
#
# They are still useful as upper limits. They show the most that could have been earned over these
# 20 days, and a strategy that can be followed, such as buy and hold, can be measured against them.

---

## Exercise 4: Saving for a first home

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/La_Sagne%2C_Maisons_Neuves_1.JPG?width=640" alt="A farmhouse in La Sagne" align="right" width="340" hspace="16">

**The plan.** A couple earning a combined 150,000 a year wants to buy their first home. Swiss
lenders require buyers to pay at least 20% of the purchase price from their own funds, so the
couple saves towards that amount.

**How the savings grow.** The savings start at 0. At the end of each month:

1. the savings earn a return of 2% per year, that is, 0.02 / 12 of the current balance;
2. then the couple adds a fixed portion of their monthly salary, which is the annual salary
   divided by 12.

**What changes each year.** After every 12 months, the salary rises by 3%, and home prices rise by
1.5%. The 20% the couple needs therefore grows too.

**The homes.** The couple considers the three homes in the list below.

<sub>Picture: a farmhouse in La Sagne, canton of Neuchâtel, photo by Elleka, <a href="https://creativecommons.org/licenses/by-sa/3.0/">CC BY-SA 3.0</a>, via Wikimedia Commons.</sub>

**What to do**

1. Write a function `months_to_save(salary, portion, price, price_growth)` that returns the number
   of months until the savings reach 20% of the home's price at that time. If this does not happen
   within 50 years, the function returns `None`.

2. Suppose the couple saves 20% of their salary. For each home, print:

   - the number of months needed with prices rising by 1.5% a year;
   - the number of months needed if prices stayed the same.

   How many extra months of saving do rising prices cost for each home?

3. The couple would like to buy within 8 years, that is, 96 months. Write a function
   `required_portion(salary, price)` that finds the portion of salary they need to save, with the
   following method, called **bisection search**:

   - the portion is searched as a whole number between 0 and 10,000, which is then divided by
     10,000, so that it has four decimals;
   - start with `low = 0` and `high = 10000`;
   - at each step, take `guess = int((low + high) / 2)`, and compute the savings and the home's
     price after 96 months when saving the portion `guess / 10000`;
   - if the savings are within 100 of 20% of that price, the search stops;
   - otherwise, if the savings are above it, set `high = guess`, and if they are below, set
     `low = guess`.

   The function returns the portion and the number of guesses it took. Before searching, it checks
   whether saving the whole salary would be enough, and returns `None` if it would not.

4. The couple cannot save more than 20% of their salary. Which of the three homes can they buy
   within 8 years?

*Adapted from Problem Set 1, "House Hunting", of MIT 6.0001 Introduction to Computer Science and
Programming in Python (Fall 2016), MIT OpenCourseWare.*

> **Hint:** saving 20% of their salary, the couple needs 80 months for the terraced house
> while prices rise, and would need 74 months if prices stayed the same.

In [16]:
salary = 150000

homes = {
    "flat in town": 750000,
    "terraced house": 1050000,
    "detached house": 1400000,
}

In [17]:
# Task 1: the function months_to_save

# The yearly rates given in the exercise.
raise_rate = 0.03              # the salary rises by 3% each year
annual_return = 0.02           # the savings earn 2% per year
yearly_price_growth = 0.015    # home prices rise by 1.5% each year


def months_to_save(salary, portion, price, price_growth):
    """Return the number of months until the savings reach 20% of the price, or None."""
    savings = 0
    months = 0

    # Repeat month after month, as long as the savings are below 20% of the current price.
    while savings < 0.20 * price:
        # At the end of the month: first the return on the savings, then the new deposit.
        monthly_return = savings * annual_return / 12
        monthly_deposit = salary / 12 * portion
        savings = savings + monthly_return + monthly_deposit

        months = months + 1

        # Every 12 months, the salary and the price both rise.
        # months % 12 is the remainder of the division by 12. It is 0 after 12, 24, 36... months.
        if months % 12 == 0:
            salary = salary * (1 + raise_rate)
            price = price * (1 + price_growth)

        # Give up after 50 years, that is 600 months.
        # Without this test, the loop would never end for a portion that is too small.
        if months > 600:
            return None

    return months


print("Task 1: check with the terraced house and 20% saved:",
      months_to_save(salary, 0.20, homes["terraced house"], yearly_price_growth), "months")

Task 1: check with the terraced house and 20% saved: 80 months


In [18]:
# Task 2: rising prices against fixed prices

print("Task 2: months of saving, with 20% of the salary saved")

for home in homes:
    price = homes[home]

    # The same calculation twice: once with prices rising by 1.5% a year, once with a growth of 0.
    rising = months_to_save(salary, 0.20, price, yearly_price_growth)
    fixed = months_to_save(salary, 0.20, price, 0)

    print()
    print("  ", home)
    print("      prices rising:", rising, "months")
    print("      prices fixed: ", fixed, "months")
    print("      extra months: ", rising - fixed)

Task 2: months of saving, with 20% of the salary saved

   flat in town
      prices rising: 58 months
      prices fixed:  55 months
      extra months:  3

   terraced house
      prices rising: 80 months
      prices fixed:  74 months
      extra months:  6

   detached house
      prices rising: 104 months
      prices fixed:  94 months
      extra months:  10


In [19]:
# Task 3: the function required_portion

def after_months(months, salary, portion, price):
    """Return the savings and the home's price after a given number of months."""
    # The same steps as in months_to_save, but for a fixed number of months.
    savings = 0

    for month in range(1, months + 1):
        monthly_return = savings * annual_return / 12
        monthly_deposit = salary / 12 * portion
        savings = savings + monthly_return + monthly_deposit

        if month % 12 == 0:
            salary = salary * (1 + raise_rate)
            price = price * (1 + yearly_price_growth)

    return {"savings": savings, "price": price}


def required_portion(salary, price):
    """Return the portion of salary to save to buy within 96 months, and the number of guesses."""
    months = 96

    # First check: even saving the whole salary (a portion of 1.0) may not be enough.
    # In that case, no search can succeed, and the loop below would never end.
    everything = after_months(months, salary, 1.0, price)
    if everything["savings"] < 0.20 * everything["price"] - 100:
        return None

    # The portion is searched as a whole number between 0 and 10,000.
    # Dividing it by 10,000 gives a portion with four decimals, for example 1642 gives 0.1642.
    low = 0
    high = 10000
    guesses = 0

    # Repeat until a guess is close enough. The return inside the loop ends the function.
    while True:
        # Try the number halfway between low and high.
        guess = int((low + high) / 2)
        guesses = guesses + 1

        result = after_months(months, salary, guess / 10000, price)
        savings = result["savings"]
        target = 0.20 * result["price"]

        # abs() gives the distance between two numbers, whichever is larger.
        if abs(savings - target) <= 100:
            return {"portion": guess / 10000, "guesses": guesses}

        if savings > target:
            # Saving this much is more than needed: the answer is below the guess.
            high = guess
        else:
            # Saving this much is not enough: the answer is above the guess.
            low = guess


print("Task 3: portion of salary needed to buy within 96 months")
print()

for home in homes:
    result = required_portion(salary, homes[home])
    print("  ", home + ":", result["portion"], "found in", result["guesses"], "guesses")

print()
print("   Check with a salary of 30,000 and the detached house:",
      required_portion(30000, homes["detached house"]))

Task 3: portion of salary needed to buy within 96 months

   flat in town: 0.1173 found in 12 guesses
   terraced house: 0.1642 found in 12 guesses
   detached house: 0.2189 found in 12 guesses

   Check with a salary of 30,000 and the detached house: None


In [20]:
# Task 4: which homes can the couple buy within 8 years?

print("Task 4: saving at most 20% of the salary")
print()

for home in homes:
    result = required_portion(salary, homes[home])

    # required_portion returns None if even the whole salary is not enough.
    if result is None:
        print("  ", home + ": no, even the whole salary is not enough")
    # Otherwise, the home is within reach if the portion needed is 20% or less.
    elif result["portion"] <= 0.20:
        print("  ", home + ": yes, it needs", result["portion"])
    else:
        print("  ", home + ": no, it needs", result["portion"])

Task 4: saving at most 20% of the salary

   flat in town: yes, it needs 0.1173
   terraced house: yes, it needs 0.1642
   detached house: no, it needs 0.2189
